# The order flow imbalance, and what it predicts

Two questions that are easily confused, and the whole of this notebook is the difference
between them.

**Does $\mathrm{OFI}$ explain the mid-price move that just happened?** Cont, Kukanov and
Stoikov show that in a stylised book it *is* that move, up to a bounded rounding term. This
is accounting, and we confirm it.

**Does $\mathrm{OFI}$ predict the next one?** That is a different question, and its answer
is not a property of $\mathrm{OFI}$. It is a property of the *kernel* that generated the
flow. We run two laboratories that differ in one structural respect and get opposite signs.

The theory is `chap.hawkes`; the code-facing half is
[`point-processes-and-hawkes.md`](../documentation/point-processes-and-hawkes.md); the
confirmatory claims and their seed blocks are fixed in
[`pre-registration-imbalance-regression.md`](../documentation/pre-registration-imbalance-regression.md).
Nothing is cached: every number is produced by the cell that reports it.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from scipy import stats

from unito26.lob import config, estimation as est, imbalance_regression as ir
from unito26.lob.benchmark import REFERENCE_PRICE
from unito26.lob.hawkes import HawkesParams
from unito26.lob.messages import GridDepth, Horizon, ReportedDepth, SweepSize, Window
from unito26.lob.orderbook import AggregateBook
from unito26.lob.session import MarketSession
from unito26.lob.simulate import EventType, OrderFlowSimulator
from unito26.lob.statistics import SessionStatistics

SURFACE, INK, MUTED, GRID_ = "#fcfcfb", "#0b0b0b", "#898781", "#e1e0d9"
pio.templates["unito26"] = go.layout.Template(layout=dict(
    paper_bgcolor=SURFACE, plot_bgcolor=SURFACE, font=dict(color=INK, size=12),
    xaxis=dict(gridcolor=GRID_, linecolor="#c3c2b7", zeroline=False, tickfont=dict(color=MUTED)),
    yaxis=dict(gridcolor=GRID_, linecolor="#c3c2b7", zeroline=False, tickfont=dict(color=MUTED)),
))
pio.templates.default = "unito26"
pd.set_option("display.width", 170)

#: The operating point, from the pre-registration.  W_STAR is fixed on the tuning block.
WARM, SAMPLE = 1200.0, 3600.0
DEPTH, PRICE_UNIT = ReportedDepth(10), 100
SPEC = SessionStatistics((GridDepth(1),), (SweepSize(100),), (1,))
BINS, PRIOR = 6, 20.0
W_STAR = {"resilient": Window(0.200), "trending": Window(0.080)}
EXPLORATORY = range(0, 6)
CONFIRMATORY = range(3000, 3012)

REGIMES = {
    "resilient": (config.example_order_flow_params(), config.example_mark_params()),
    "trending": (config.trending_order_flow_params(), config.trending_mark_params()),
}
PRESSURE = np.array([event.pressure for event in EventType], dtype=float)


#: A session depends on the regime and the seed and on nothing else, and building one
#: is the expensive step here.  Every ladder below reads the same sessions at
#: different windows, so they are built once.
_SESSIONS: dict = {}


def build(regime, seed, flow=None):
    """One warmed session, cached.  Warmed for WARM, then sampled over the next SAMPLE."""
    if flow is None and (regime, seed) in _SESSIONS:
        return _SESSIONS[regime, seed]
    default, marks = REGIMES[regime]
    simulator = OrderFlowSimulator(flow if flow is not None else default, marks,
                                   REFERENCE_PRICE, rng=seed)
    book = AggregateBook()
    simulator.warm_up(book, horizon=WARM, journal=None)
    opening, messages = book.copy(), []
    for message in simulator.stream(book, horizon=WARM + SAMPLE, journal=None):
        book.apply(message, record=False)
        messages.append(message)
    session = MarketSession.from_occupied_levels(
        opening, messages, DEPTH, SPEC, PRICE_UNIT, True)
    if flow is None:
        _SESSIONS[regime, seed] = session
    return session


def pieces(session, window):
    """Mid, flow, OFI over `window`, and the queue imbalance, on one clock."""
    mid, flow = ir.aligned_mid_price(session), ir.aligned_order_flow(session)
    return dict(
        mid=mid, flow=flow,
        ofi=ir.backward_sum(flow, window),
        imbalance=np.where(
            session.stats["QueueImbalance1Covered"].to_numpy(float) > 0,
            session.stats["QueueImbalance1"].to_numpy(float), np.nan),
    )


print(f"{'regime':>10} {'rho':>6} {'nu':>8} {'signed endogenous':>19} {'depth decay':>12}")
for name, (flow, marks) in REGIMES.items():
    print(f"{name:>10} {flow.branching_ratio:6.3f} {flow.stationary_intensity().sum():8.3f}"
          f" {flow.signed_endogenous_fraction(PRESSURE):19.3f} {marks.depth_decay:12.2f}")

    regime    rho       nu   signed endogenous  depth decay
 resilient  0.600   30.188              -0.351         0.08
  trending  0.600   30.188               0.554         0.15


## 1. Cont, Kukanov and Stoikov: $\mathrm{OFI}$ *is* the mid-price move

Take a book in which limit orders and cancellations arrive only at the best quote, and every
level beyond the best holds the same depth $D$ on both sides. Then over a window the number
of ticks the best bid moves is the net buy-side flow divided by $D$, rounded; likewise the
ask; and the mid moves by half their sum. Writing $\tau$ for the tick,

$$\Delta P^m \;=\; \tau\left[\frac{\mathrm{OFI}}{2D} \;+\; \varepsilon\right],
\qquad \varepsilon = \tfrac12\big[(\lceil x\rceil - x) - (\lceil y\rceil - y)\big] \in (-\tfrac12, \tfrac12).$$

$\mathrm{OFI}/(2D)$ is shares over shares, so it is dimensionless: **the left-hand side is a
tick count**, which is how this package carries prices throughout.

This is not a forecast. It is an identity about the window that has already happened, and
the residual is bounded by half a tick *inside the model*. Both of the model's hypotheses
fail here by construction — limit orders rest at geometrically decaying offsets rather than
only at the touch, and the depth profile decays rather than being flat — so we should not
expect the identity, only its shadow. We measure how much of it survives.

In [2]:
windows = (0.1, 0.325, 1.0, 3.0, 10.0)
rows = []
for regime in REGIMES:
    per_window = {w: [] for w in windows}
    for seed in EXPLORATORY:
        session = build(regime, seed)
        for window in windows:
            part = pieces(session, Window(window))
            move = ir.backward_change(part["mid"], Window(window))
            defined = np.isfinite(part["ofi"]) & np.isfinite(move)
            per_window[window].append(np.corrcoef(part["ofi"][defined], move[defined])[0, 1])
    for window in windows:
        rows.append({"regime": regime, "window (s)": window,
                     "corr(OFI, dP over the same window)": np.mean(per_window[window]),
                     "sd across seeds": np.std(per_window[window], ddof=1)})

contemporaneous = pd.DataFrame(rows)
print(contemporaneous.to_string(index=False, float_format=lambda v: f"{v:+.4f}"))
print("\nPositive in both regimes, and rising with the window: exactly what a residual")
print("bounded by half a tick predicts, since Var(dP) grows with the window and Var(eps)")
print("does not.  This much is accounting, and it does not distinguish the two laboratories.")

   regime  window (s)  corr(OFI, dP over the same window)  sd across seeds
resilient     +0.1000                             +0.3808          +0.0079
resilient     +0.3250                             +0.4083          +0.0106
resilient     +1.0000                             +0.4475          +0.0103
resilient     +3.0000                             +0.5008          +0.0130
resilient    +10.0000                             +0.5583          +0.0301
 trending     +0.1000                             +0.2961          +0.0255
 trending     +0.3250                             +0.3583          +0.0257
 trending     +1.0000                             +0.4612          +0.0290
 trending     +3.0000                             +0.5433          +0.0355
 trending    +10.0000                             +0.6101          +0.0417

Positive in both regimes, and rising with the window: exactly what a residual
bounded by half a tick predicts, since Var(dP) grows with the window and Var(eps)
does not.  Thi

## 2. The two statistics

**The order flow contribution.** For each event $n$, $e_n$ is the *change in size at the
touch* attributable to it, signed by which side it helps:

$$e_n = \mathbb 1\{P^b_n \ge P^b_{n-1}\}S^b_n - \mathbb 1\{P^b_n \le P^b_{n-1}\}S^b_{n-1}
      - \mathbb 1\{P^a_n \le P^a_{n-1}\}S^a_n + \mathbb 1\{P^a_n \ge P^a_{n-1}\}S^a_{n-1}.$$

It is **not** the signed size of the order. A limit order resting ten ticks behind the touch
has $e_n = 0$; a cancellation at the touch contributes a whole queue. That gap is where
$\mathrm{OFI}$ stops being a signed event count and starts reading the book — and it is the
reason the marks matter and the kernel alone cannot reproduce it.

$$\mathrm{OFI}_{t,w} = \sum_{n\,:\,t-w < T_n \le t} e_n,
\qquad
I^n_t = \frac{\sum_{i\le n} S^{b,i}_t - \sum_{i \le n} S^{a,i}_t}{\sum_{i\le n} S^{b,i}_t + \sum_{i\le n} S^{a,i}_t}.$$

$\mathrm{OFI}$ is a *flow* over a window and unbounded; $I^n$ is a *state* at an instant and
lies in $[-1,1]$. Bid-heavy is positive for both.

In [3]:
session = build("resilient", 0)
part = pieces(session, W_STAR["resilient"])
frame = pd.DataFrame({
    "e_n": part["flow"].values,
    f"OFI over {float(W_STAR['resilient'])}s": part["ofi"],
    "I^1": part["imbalance"],
    "next dP^m": ir.forward_change_to_next_event(part["mid"]),
})
print(frame.describe().T.to_string(float_format=lambda v: f"{v:.4f}"))
print(f"\nrows {len(frame):,}   e_n exactly zero on {np.mean(part['flow'].values == 0)*100:.1f}% of events,")
print("which is every limit order and cancellation that did not touch the best quote.")

                    count    mean      std        min      25%     50%     75%       max
e_n           109477.0000  0.1782  41.1667 -1040.0000   0.0000  0.0000  0.0000 1160.0000
OFI over 0.2s 109467.0000  1.4072 122.7785 -1100.0000 -50.0000  0.0000 50.0000 1270.0000
I^1           109478.0000 -0.0189   0.5832    -0.9882  -0.5556 -0.0149  0.5111    0.9879
next dP^m     109477.0000  0.0004   0.3504    -5.0000   0.0000  0.0000  0.0000    6.0000

rows 109,478   e_n exactly zero on 76.7% of events,
which is every limit order and cancellation that did not touch the best quote.


## 3. The obstacle: the outcome is a lattice with an atom

At the next-event horizon the mid usually does not move at all, and when it moves it moves by
a half-tick multiple. So $\Delta P^m$ is not a continuous variable with a bit of noise: it is
a point mass at zero plus a sparse lattice. Two of the reflexes one reaches for fail on it,
and they fail for reasons worth separating.

$R^2$ is small because the *variance* is concentrated in rare rows, not because the
relationship is weak. Sign accuracy is high but uninformative, because predicting "no move"
is right most of the time. The question is whether **linearity** is what is binding, and for
that we compare the linear fit against the best any function of the predictor could do on the
same partition — the conditional mean, in $R^2$ terms.

In [4]:
part = pieces(session, W_STAR["resilient"])
outcome = ir.forward_change_to_next_event(part["mid"])
defined = np.isfinite(part["ofi"]) & np.isfinite(outcome)
x, y = part["ofi"][defined], outcome[defined]

K = 20
design = np.column_stack([np.ones(x.size), x])
linear = est.ols(design, y)
edges = np.unique(np.quantile(x, np.linspace(0, 1, K + 1)[1:-1]))
cell = np.searchsorted(edges, x, side="right")
means = np.array([y[cell == c].mean() if np.any(cell == c) else y.mean() for c in range(cell.max() + 1)])
ceiling = 1 - np.sum((y - means[cell]) ** 2) / np.sum((y - y.mean()) ** 2)
floor = (cell.max()) / (x.size - 1)

print(f"P(dP = 0)                       {np.mean(y == 0):.4f}")
print(f"realised support (ticks)        {np.unique(y)[:3]} ... {np.unique(y)[-3:]}")
print(f"max |dP|                        {np.abs(y).max():.1f}")
print(f"share of sum(dP^2) in top 0.1%  {np.sort(y**2)[-int(0.001*y.size):].sum() / np.sum(y**2):.3f}")
print(f"participation ratio of dP       {est.participation_ratio(y):.1f} of {y.size:,} rows")
print(f"participation ratio of OFI      {est.participation_ratio(x):.1f}")
print(f"Hill tail index of |OFI|        {est.hill_tail_index(x, max(2, x.size // 100)):.2f}   (not computed on dP:")
print( "                                a lattice ties most tail rows to the threshold)")
print()
print(f"linear R^2                      {linear.r_squared:.5f}")
print(f"nonparametric ceiling, k = {K}    {ceiling:.5f}   null floor (k-1)/(n-1) = {floor:.5f}")
print(f"ratio                           {linear.r_squared / ceiling:.3f}")
print()
print("The linear fit reaches most of the ceiling, so linearity was never the binding")
print("constraint -- but both numbers are of order 1e-3, and the honest reading is that")
print("R^2 is the wrong instrument here rather than that the relation is absent.")

P(dP = 0)                       0.9115
realised support (ticks)        [-5.  -4.5 -4. ] ... [4.5 5.  6. ]
max |dP|                        6.0
share of sum(dP^2) in top 0.1%  0.120
participation ratio of dP       2629.8 of 109,466 rows
participation ratio of OFI      11237.5
Hill tail index of |OFI|        4.14   (not computed on dP:
                                a lattice ties most tail rows to the threshold)

linear R^2                      0.00037
nonparametric ceiling, k = 20    0.00062   null floor (k-1)/(n-1) = 0.00014
ratio                           0.592

The linear fit reaches most of the ceiling, so linearity was never the binding
constraint -- but both numbers are of order 1e-3, and the honest reading is that
R^2 is the wrong instrument here rather than that the relation is absent.


## 4. Three ways round it, ordered by how much each concedes

**Coarsen the clock.** Sum the flow and difference the mid over fixed buckets. This concedes
resolution and nothing else, and it is the regime the mid starts to look like a diffusion
rather than a step function.

**Change the outcome.** Count up-moves and down-moves separately in $(t, t+h]$: a window
holding one of each is then distinguishable from a quiet one, which the net change cannot do.
Or use the forward VWAP — an open experiment, with no position taken in advance.

**Change the metric.** Score a three-class model by log score against climatology, which is
lattice-native; and measure dependence directly with mutual information, which assumes
nothing about the functional form.

In [5]:
buckets = np.array([0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 60.0])
rows = []
for regime in REGIMES:
    per_bucket = {bucket: [] for bucket in buckets}
    for seed in EXPLORATORY:
        part = pieces(build(regime, seed), W_STAR[regime])
        cumulative = ir.AlignedSeries(
            part["flow"].times, np.nan_to_num(part["flow"].values).cumsum(), part["flow"].segments)
        for bucket in buckets:
            coarse_mid = ir.bucket_series(part["mid"], Horizon(float(bucket)))
            coarse_flow = ir.bucket_series(cumulative, Horizon(float(bucket)))
            move, flow_in = np.diff(coarse_mid.values), np.diff(coarse_flow.values)
            # The trending book empties, so its mid is undefined on some rows and the
            # difference across them is not a price change.  Drop those pairs.
            pair = np.isfinite(flow_in[:-1]) & np.isfinite(move[1:])
            if pair.sum() > 3:
                per_bucket[bucket].append(
                    np.corrcoef(flow_in[:-1][pair], move[1:][pair])[0, 1])
    for bucket in buckets:
        rows.append({"regime": regime, "bucket (s)": bucket,
                     "rows/session": int(SAMPLE / bucket),
                     "corr(OFI in bucket, next bucket's move)": np.mean(per_bucket[bucket])})

ladder = pd.DataFrame(rows)
print(ladder.to_string(index=False, float_format=lambda v: f"{v:+.4f}"))
print(f"\nT/b >= 50 is the grid rule, so b <= {SAMPLE/50:.0f} s on one session; beyond that the")
print("row count is too small to read and pooling across seeds is required.  The predictive")
print("ladder is the informative one -- a contemporaneous R^2 rises with the bucket by")
print("arithmetic, since Var(dP) grows while the bounded residual does not.")

   regime  bucket (s)  rows/session  corr(OFI in bucket, next bucket's move)
resilient     +0.1000         36000                                  -0.0243
resilient     +0.3000         12000                                  -0.0458
resilient     +1.0000          3600                                  -0.0825
resilient     +3.0000          1200                                  -0.1451
resilient    +10.0000           360                                  -0.1761
resilient    +30.0000           120                                  -0.1466
resilient    +60.0000            60                                  -0.0913
 trending     +0.1000         36000                                  +0.0212
 trending     +0.3000         12000                                  +0.0370
 trending     +1.0000          3600                                  +0.0063
 trending     +3.0000          1200                                  -0.0093
 trending    +10.0000           360                                  +0.0302

In [6]:
# Changing the outcome.  Jump counts are defined on every row; the forward VWAP is not,
# and its missingness is the first thing to report because it is not missing at random.
horizons = np.array([0.05, 0.1, 0.325, 1.0, 3.0])
gathered = {h: {"up": [], "down": [], "traded": []} for h in horizons}
for seed in EXPLORATORY:
    session = build("resilient", seed)
    part = pieces(session, W_STAR["resilient"])
    for horizon in horizons:
        up, down = ir.forward_jump_counts(part["mid"], Horizon(float(horizon)))
        vwap = ir.forward_vwap(session, Horizon(float(horizon)))
        both = np.isfinite(part["ofi"]) & np.isfinite(up)
        gathered[horizon]["up"].append(np.corrcoef(part["ofi"][both], up[both])[0, 1])
        gathered[horizon]["down"].append(np.corrcoef(part["ofi"][both], down[both])[0, 1])
        gathered[horizon]["traded"].append(np.mean(np.isfinite(vwap)))

print(pd.DataFrame([
    {"horizon (s)": h, "corr(OFI, up count)": np.mean(g["up"]),
     "corr(OFI, down count)": np.mean(g["down"]), "windows that traded": np.mean(g["traded"])}
    for h, g in gathered.items()
]).to_string(index=False, float_format=lambda v: f"{v:+.4f}"))
print("\nThe two counts move oppositely, which the net change cannot show.")
print("The forward VWAP is undefined on most short windows, and a window trades exactly when")
print("the clustered market-order flow that also drives |OFI| fires -- so the missingness is")
print("not at random and a complete-case mean would condition on activity.  It is analysable")
print("only where the trading rate is near one, and the rate is reported first for that reason.")

 horizon (s)  corr(OFI, up count)  corr(OFI, down count)  windows that traded
     +0.0500              -0.0090                +0.0084              +0.2281
     +0.1000              -0.0091                +0.0111              +0.3836
     +0.3250              -0.0135                +0.0118              +0.7157
     +1.0000              -0.0181                +0.0103              +0.9411
     +3.0000              -0.0239                +0.0148              +0.9989

The two counts move oppositely, which the net change cannot show.
The forward VWAP is undefined on most short windows, and a window trades exactly when
the clustered market-order flow that also drives |OFI| fires -- so the missingness is
not at random and a complete-case mean would condition on activity.  It is analysable
only where the trading rate is near one, and the rate is reported first for that reason.


## 5. The result: the sign belongs to the kernel

Everything so far has been exploratory, on seeds 0–5 and reported without inference. This
section is the confirmatory run: **seeds 3000–3011**, disjoint from exploration and from the
block on which $w^*$ was tuned, and read by nothing before it was run. The window, the
horizon, the bin count and the fold split were all fixed beforehand in the pre-registration.

Four claims, Holm-corrected within the family at family-wise 5%. The fourth is the one this
notebook is about.

In [7]:
def confirmatory(regime):
    window, purge = W_STAR[regime], int(np.ceil(float(W_STAR[regime]) * 30.19))
    out = {"D1": [], "D2": [], "D3": [], "signed covariance": [], "masked": []}
    for seed in CONFIRMATORY:
        s = build(regime, seed)
        p = pieces(s, window)
        nxt = ir.forward_change_to_next_event(p["mid"])
        keep = (np.isfinite(p["ofi"]) & np.isfinite(nxt)
                & np.isfinite(p["flow"].values) & np.isfinite(p["imbalance"]))
        out["masked"].append(1 - keep.mean())
        y = ir.outcome_classes(nxt[keep])
        base, ofi, qi = p["flow"].values[keep], p["ofi"][keep], p["imbalance"][keep]

        def skill(a, b):
            half = a.size // 2
            return est.incremental_log_score_skill(
                a[:half - purge], b[:half - purge], y[:half - purge],
                a[half:], b[half:], y[half:], BINS, PRIOR)

        out["D1"].append(skill(base, ofi - base))
        out["D2"].append(skill(qi, ofi))
        out["D3"].append(skill(ofi, qi) - out["D2"][-1])
        ahead = ir.forward_change(p["mid"], Horizon(1.0))
        ok = np.isfinite(p["ofi"]) & np.isfinite(ahead)
        out["signed covariance"].append(float(ir.signed_covariance(p["ofi"][ok], ahead[ok])))
    return {k: np.array(v) for k, v in out.items()}


results = {regime: confirmatory(regime) for regime in REGIMES}

claims, pvalues = [], {}
for name, label in (("D1", "C1  the window adds over the last event alone"),
                    ("D2", "C2  OFI adds over I^1"),
                    ("D3", "C3  I^1 adds more over OFI than OFI over I^1")):
    v = results["resilient"][name]
    t, p = stats.ttest_1samp(v, 0.0)
    p = p / 2 if t > 0 else 1 - p / 2
    pvalues[label] = p
    claims.append({"claim": label, "mean": v.mean(), "sd": v.std(ddof=1),
                   "seeds on side": f"{int((v > 0).sum())}/12", "one-sided p": p})

up, down = results["trending"]["signed covariance"], results["resilient"]["signed covariance"]
_, p_contrast = stats.ttest_ind(up, down)
_, p_up = stats.ttest_1samp(up, 0.0)
_, p_down = stats.ttest_1samp(down, 0.0)
compound = max(p_contrast / 2, p_up / 2, p_down / 2)
label = "C4  the predictive sign is opposite between the regimes"
pvalues[label] = compound
claims.append({"claim": label, "mean": up.mean() - down.mean(),
               "sd": np.sqrt(up.var(ddof=1) + down.var(ddof=1)),
               "seeds on side": f"{int((up > 0).sum())}/12 and {int((down < 0).sum())}/12",
               "one-sided p": compound})

print(pd.DataFrame(claims).to_string(index=False, float_format=lambda v: f"{v:+.3e}"))
print()
ordered = sorted(pvalues.items(), key=lambda kv: kv[1])
survives = True
for i, (label, p) in enumerate(ordered):
    threshold = 0.05 / (len(ordered) - i)
    survives = survives and p <= threshold
    print(f"Holm  {label:<52}  p {p:.2e} <= {threshold:.4f}   {'REJECT' if survives else 'retain'}")
print()
for regime in REGIMES:
    print(f"{regime:>10}: Cov(sign OFI, dP over 1s) = {results[regime]['signed covariance'].mean():+.5f}"
          f"   masked {results[regime]['masked'].mean()*100:.4f}%")

                                                  claim       mean         sd   seeds on side  one-sided p
          C1  the window adds over the last event alone +2.422e-03 +6.979e-04           12/12   +5.706e-08
                                  C2  OFI adds over I^1 +1.347e-03 +4.658e-04           12/12   +3.626e-07
           C3  I^1 adds more over OFI than OFI over I^1 +1.658e-02 +1.415e-03           12/12   +1.236e-13
C4  the predictive sign is opposite between the regimes +1.075e-01 +1.308e-02 12/12 and 12/12   +3.444e-05

Holm  C3  I^1 adds more over OFI than OFI over I^1          p 1.24e-13 <= 0.0125   REJECT
Holm  C1  the window adds over the last event alone         p 5.71e-08 <= 0.0167   REJECT
Holm  C2  OFI adds over I^1                                 p 3.63e-07 <= 0.0250   REJECT
Holm  C4  the predictive sign is opposite between the regimes  p 3.44e-05 <= 0.0500   REJECT

 resilient: Cov(sign OFI, dP over 1s) = -0.08823   masked 0.0074%
  trending: Cov(sign OFI, dP over 

In [8]:
# The control that pins the cause.  rho = 0 removes the kernel and holds lambda* exactly,
# so the flow is Poisson of the same composition and the same rate.
control = {}
for regime, (flow, _) in REGIMES.items():
    poisson = HawkesParams(flow.stationary_intensity(), np.zeros((6, 6)), flow.decay)
    per_seed = []
    for seed in CONFIRMATORY:
        s = build(regime, seed, flow=poisson)
        p = pieces(s, W_STAR[regime])
        ahead = ir.forward_change(p["mid"], Horizon(1.0))
        ok = np.isfinite(p["ofi"]) & np.isfinite(ahead)
        per_seed.append(float(ir.signed_covariance(p["ofi"][ok], ahead[ok])))
    control[regime] = np.array(per_seed)

summary = pd.DataFrame({
    "kernel on": {r: results[r]["signed covariance"].mean() for r in REGIMES},
    "rho = 0 control": {r: control[r].mean() for r in REGIMES},
    "control sd": {r: control[r].std(ddof=1) for r in REGIMES},
})
print(summary.to_string(float_format=lambda v: f"{v:+.5f}"))
print()
print("Read this table carefully, because it says something the two-regime table alone")
print("does not.  With rho = 0 there is no excitation at all and the flow is Poisson of")
print("the same composition -- yet the forward sign is *negative in both*.  Reversal is")
print("therefore mechanical: it is the book refilling a queue that was cleared, and it")
print("is there before any kernel is.")
print()
print("So a kernel does not create the forward sign; it can only reinforce that")
print("mechanical reversal or overcome it.  The resilient kernel is aligned with it and")
print("adds nothing measurable.  The trending kernel works against it, and wins: it")
print("moves the statistic by about +0.041, which is more than the -0.021 the book")
print("supplies on its own, and the sign flips.  That is the whole of the effect in C4.")

           kernel on  rho = 0 control  control sd
resilient   -0.08823         -0.08879    +0.01110
trending    +0.01923         -0.02134    +0.00337

Read this table carefully, because it says something the two-regime table alone
does not.  With rho = 0 there is no excitation at all and the flow is Poisson of
the same composition -- yet the forward sign is *negative in both*.  Reversal is
therefore mechanical: it is the book refilling a queue that was cleared, and it
is there before any kernel is.

So a kernel does not create the forward sign; it can only reinforce that
mechanical reversal or overcome it.  The resilient kernel is aligned with it and
adds nothing measurable.  The trending kernel works against it, and wins: it
moves the statistic by about +0.041, which is more than the -0.021 the book
supplies on its own, and the sign flips.  That is the whole of the effect in C4.


### 5b. Adverse selection, and why this laboratory does not show it

No code in this section. It is the most important caveat on the numbers above, and it is an
argument rather than a measurement.

**What adverse selection is.** In a traded market some participants know something the
others do not. A liquidity provider resting size on the bid has written an option: his order
is filled precisely when someone wants to sell, which is disproportionately when selling
turns out to have been right. He is picked off, and the fill is worst exactly when it is most
likely. His defence is to withdraw before it happens.

**What that does to the book.** It means a queue that is about to be picked off is a queue
whose own side pulls away from it. Depletion begets depletion. And it is why the queue
imbalance $I^n$ is informative in a real book: a thin bid is not merely *mechanically*
closer to being cleared, it is thin **because** the providers on that side inferred
something and acted. The imbalance is reading their inference. That is an informational
channel, and it is most of what makes $I^n$ a signal rather than a counter.

**Why we do not see it here.** Our generator has no informed trader, so there is nothing to
infer and no one to be picked off by. Worse — or rather, more deliberately — its kernel does
the *opposite*. It is built so that whatever depletes a side excites the limit orders that
refill it, nineteen times more strongly than it excites further withdrawals. That
replenishment mechanism is precisely the reverse of an adverse-selection channel, and the
two cannot both be strong in the same kernel: one says depletion calls forth supply, the
other says depletion calls forth flight. We chose the replenishing one, because a book that
survives is the precondition for measuring anything at all, and a book with a strong
adverse-selection channel and no informed trader to justify it simply empties.

**What that says about the queue imbalance's forecasting power.** Whatever $I^1$ achieves in
this notebook is **mechanical**: a depleted queue is one market order away from clearing, and
clearing moves the mid. None of it is informational. So this laboratory measures $I^1$ at its
weakest.

Read C3 in that light. It says $I^1$ adds an order of magnitude more over $\mathrm{OFI}$ than
$\mathrm{OFI}$ adds over $I^1$ — and it says so with the imbalance's principal advantage
switched off. It is therefore a **lower bound** on the imbalance's advantage, not an estimate
of it. The same asymmetry governs every comparison here: a result favourable to
$\mathrm{OFI}$ is not evidence that $\mathrm{OFI}$ dominates the imbalance in general, while
a result favourable to $I^1$ is the stronger finding, because it was obtained against a
handicap.

The honest summary is that we have measured the *mechanical* content of both statistics
cleanly, and the *informational* content of neither.

## 6. Findings, and what they are not

**What was established.** In both laboratories $\mathrm{OFI}$ is the mechanical update of the
mid: contemporaneously it correlates with the move over the same window at $+0.36$ to $+0.56$,
rising with the window as a residual bounded by half a tick predicts. That much is Cont,
Kukanov and Stoikov's accounting, and it survives the failure of both their hypotheses.

Forward, all four pre-registered claims confirm on twelve unread seeds, each with every seed
on the claimed side. A window of recent flow adds information over the last event alone. It
adds information over the queue imbalance. The queue imbalance adds an order of magnitude
more over it. And — the point of the notebook — **the sign of what $\mathrm{OFI}$ predicts is
opposite in the two regimes**, positive where excitation follows the pressure partition and
negative where it crosses it.

**What that means.** The predictive sign is not a property of $\mathrm{OFI}$. It is a
property of the kernel, read through $\mathrm{OFI}$. Ask "does order flow imbalance predict
continuation or reversal?" and the answer is a question in return: in which market? The
contemporaneous relation is accounting and is the same in both; the forward relation is the
regime.

The $\rho = 0$ control sharpens this rather than simply confirming it. With no excitation at
all the forward sign is **negative in both regimes**, so reversal is mechanical — the book
refilling a queue that was cleared — and it is there before any kernel is. A kernel cannot
create the forward sign; it can only reinforce that reversal or overcome it. The resilient
kernel is aligned with it and adds nothing measurable; the trending kernel works against it
by about $+0.04$, which is enough to flip it. C4 is that flip.

**What it is not.** A result about two estimators applied to a generator whose parameters we
chose, read against that generator's own dimensionless groups. Not evidence about traded
markets. Specifically:

- there is **no adverse selection** here, by construction and in the strong sense of §5b, so
  $I^1$ is measured at its weakest and every comparison must be read asymmetrically;
- the intensities do not read the book, so there is no queue-depletion feedback; only the
  marks see it;
- one excitation timescale, one exponential kernel, no intraday non-stationarity;
- no anchor for the price level, so the per-seed drift is a random effect of the size of the
  effects reported, which is why pooled rows are demeaned within seed;
- the trending regime empties a side on about 0.2% of rows. That is not a defect but the
  regime: a book whose flow is momentum-carrying is a book whose liquidity gets stripped.
  It is why the two regimes carry different survival gates.

Two questions cannot be asked in this setting at all, because they need recorded data:
whether either regime occurs in a traded market, and whether one excitation timescale
suffices to describe one.